# Gene Expression Programs: Interpretation

**REQUIRED DAY 3**

## Loading the instructor's real consensus output

K=7, selected in lesson 07 by highest stability (silhouette=0.917) across the full 6-K x 100-iteration grid. Two matrices: **usage** (how much each cell relies on each program) and **spectra** (which genes define each program).

In [ ]:
import pandas as pd
import scanpy as sc

GRID_DIR = "/tscc/nfs/home/juf009/day3_shared_data/cd4_full_grid"

usage = pd.read_csv(f"{GRID_DIR}/cd4_full_grid.usages.k_7.dt_0_5.consensus.txt", sep="\t", index_col=0)
spectra = pd.read_csv(f"{GRID_DIR}/cd4_full_grid.gene_spectra_score.k_7.dt_0_5.txt", sep="\t", index_col=0)
print("usage (cells x programs):", usage.shape)
print("spectra (programs x genes):", spectra.shape)

## What is each program actually made of?

For each program, the genes with the highest spectra score are the ones defining it — read them the way you'd read `rank_genes_groups` output, not by vibes.

In [ ]:
for program in spectra.index:
    top_genes = spectra.loc[program].sort_values(ascending=False).head(10)
    print(f"Program {program}:", list(top_genes.index))

## Name them yourself before reading further

Real output from this exact run:

- **Program 1**: `RPS6, RPL7, RPS3A, RPL10, RPL13, RPL3, RPL21, RPS4X, RPL10A, RPS18` — every single gene is a ribosomal protein (RPS/RPL). A **translation/housekeeping program**, present in every cell to some degree.
- **Program 3**: `ISG15, IFIT3, ISG20, IFI6, IFIT1, MX1, LY6E, OAS1, TNFSF10, IFIT2` — the same interferon-stimulated genes from lessons 04 and 06. **cNMF found the interferon response with zero knowledge of which cells were stimulated** — it doesn't see the `label` column at all. This is an unsupervised confirmation of a supervised result, through a completely different method.
- **Program 4**: `HSP90AB1, UBB, HSPA8, HSPB1, UBC, HSPE1, HSPH1, DNAJB6, HSP90AA1` — heat shock / protein folding stress response.
- **Program 6**: `SDPR, TUBB1, PF4, PPBP, ACRBP, GNG11, RGS18, PTCRA, GP9, TMEM40` — **every one of these is a platelet gene**, inside a dataset annotated as "CD4 T cells." This is not biology — platelet fragments and free platelet mRNA are a well-known contaminant in blood scRNA-seq. cNMF doesn't know the difference between a real activation program and a technical artifact; you have to.

## Does program usage actually shift with stimulation?

Programs were discovered blind to condition — now bring `label` back in and check.

In [ ]:
adata = sc.read_h5ad("/tscc/nfs/home/juf009/day3_shared_data/kang_2018_checkpoint.h5ad")
sub_obs = adata.obs.loc[adata.obs["cell_type"] == "CD4 T cells", ["label", "replicate"]]

usage_norm = usage.div(usage.sum(axis=1), axis=0)
usage_norm.columns = [f"program_{c}" for c in usage_norm.columns]
usage_norm = usage_norm.reindex(sub_obs.index)
combined = usage_norm.join(sub_obs)

combined.groupby("label", observed=True)[usage_norm.columns].mean()

Real numbers: **Program 3 (interferon) usage jumps from 0.015 (ctrl) to 0.323 (stim)** — over a 20-fold increase, found without ever telling cNMF which cells were stimulated. Program 1 (ribosomal) drops from 0.570 to 0.364 — plausible: cells reallocating away from baseline translation toward mounting an interferon response. Program 6 (the platelet artifact) barely moves (0.0082 -> 0.0074) — consistent with it being a technical contaminant unrelated to the biological question, not a real condition effect.

## Compare against your own toy run

Open your own `results/cnmf_practice/cd4_toy_run/` output from lesson 07 (10 iterations, vs. the instructor's 100). Load your own K=7 spectra file the same way and compare its top genes per program against the instructor's list above — expect yours to be noisier, possibly with a program that doesn't cleanly match any of the seven above. That gap is Agent-B checklist item 23, made visible.

## Agent-assisted GEP interpretation, done right

> Weak: "What do these gene programs mean?"
>
> Strong: "For each program, list its top 10 genes by spectra score and tell me if you recognize a coherent biological function (e.g., ribosomal, interferon response, heat shock) or flag it as unclear — don't force a biological story onto a program whose top genes don't obviously cohere."

## Practice

Run the usage-by-condition comparison yourself. Open a fresh Agent B session and run checklist item 23 against your own toy run vs. the instructor's full run.

## Further reading

- [Kotliar et al. 2019, eLife](https://elifesciences.org/articles/43803)
- [cNMF documentation](https://github.com/dylkot/cNMF)